# 01 — Esplorazione Hi-C end-to-end (hg38, chr2)

Dal file `.mcool` alle analisi principali, sempre **per regione** (mai un cromosoma intero ad alta risoluzione in RAM):

1. risoluzioni disponibili
2. vista d'insieme di chr2 a bassa risoluzione
3. zoom su 10 Mb e confronto tra due regioni
4. curva P(s)
5. compartimenti A/B (E1 orientato con il GC di hg38)
6. insulation score e confini di TAD
7. vista circolare (Circos) dei contatti tra cromosomi
8. salvataggio dei risultati in `results/`

**Dati**: scarica un `.mcool` hg38 bilanciato dal 4DN Data Portal (GM12878: `make mcool ACC=4DNFIXP4QG5B`, 27 GB, oppure `ACC=4DNFIXVAKX9Q`, 3,9 GB; vedi README) e imposta `MCOOL` qui sotto.
Se il file non c'è, il notebook usa un **dataset sintetico** con TAD e compartimenti noti, così tutto gira lo stesso.

In [ ]:
import sys
from pathlib import Path

# radice del progetto, sia in Docker (/work) sia in locale (notebooks/ -> ..)
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
import logging
import warnings

import cooler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from clew import analysis, hic_io
from clew.circos import cis_links, plot_circos, trans_links
from clew.synthetic import make_synthetic_mcool, true_compartment

RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

# rumore interno a cooltools 0.7 con pandas 2.x, non riguarda i risultati
warnings.filterwarnings("ignore", category=FutureWarning, module="cooltools")
logging.getLogger().setLevel(logging.WARNING)

## Configurazione

In [ ]:
# GM12878 hg38 dal 4DN: il primo che esiste (`make mcool ACC=...`)
CANDIDATES = ["4DNFIXP4QG5B.mcool",   # 27,4 GB, MboI, protocollo di riferimento (Rao 2014)
              "4DNFIXVAKX9Q.mcool"]   # 3,9 GB, DpnII
MCOOL = next((ROOT / "data" / f for f in CANDIDATES if (ROOT / "data" / f).exists()),
             ROOT / "data" / CANDIDATES[0])
FASTA = ROOT / "data" / "genomes" / "hg38" / "hg38.fa"  # da `make genome`
NPROC = 4
CHROM = "chr2"

DEMO = not MCOOL.exists()
if not DEMO:
    RES = dict(overview=250_000, zoom=10_000, compartments=100_000, insulation=10_000, circos=1_000_000)
    ZOOM = "chr2:10,000,000-20,000,000"
    PAIR = ("chr2:10,000,000-15,000,000", "chr2:150,000,000-155,000,000")
    INS_REGION = "chr2:10,000,000-20,000,000"
else:
    print(f"{MCOOL.name} non trovato: uso il dataset sintetico (TAD 200 kb, compartimenti 400 kb)")
    synth = RESULTS / "synthetic" / "synthetic.mcool"
    MCOOL = synth if synth.exists() else Path(make_synthetic_mcool(synth.parent))
    RES = dict(overview=40_000, zoom=10_000, compartments=40_000, insulation=10_000, circos=40_000)
    ZOOM = "chr2:0-1,000,000"
    PAIR = ("chr2:0-800,000", "chr2:800,000-1,600,000")
    INS_REGION = "chr2"

print("file:", MCOOL.relative_to(ROOT))

## 1. Risoluzioni disponibili

In [ ]:
resolutions = hic_io.list_resolutions(MCOOL)
missing = {k: r for k, r in RES.items() if r not in resolutions}
assert not missing, f"risoluzioni mancanti nel file: {missing}"

## 2. Vista d'insieme del cromosoma

Bassa risoluzione (250 kb in hg38: chr2 corrisponde a circa 970 bin, pochi MB di RAM). Si vedono la scacchiera dei compartimenti e il decadimento lungo la diagonale.

In [ ]:
clr_ov = hic_io.open_cooler(MCOOL, RES["overview"])
chrom_region = (CHROM, 0, int(clr_ov.chromsizes[CHROM]))
mat_ov = hic_io.fetch_region(MCOOL, RES["overview"], chrom_region)
print(f"{CHROM} @ {RES['overview']:,} bp: matrice {mat_ov.shape}, {mat_ov.nbytes / 1e6:.1f} MB")
hic_io.plot_matrix(mat_ov, chrom_region, title=f"{CHROM} @ {RES['overview'] // 1000} kb");

## 3. Zoom e confronto tra regioni

In [ ]:
mat_zoom = hic_io.fetch_region(MCOOL, RES["zoom"], ZOOM)
print(f"{ZOOM} @ {RES['zoom']:,} bp: matrice {mat_zoom.shape}, {mat_zoom.nbytes / 1e6:.1f} MB")
hic_io.plot_matrix(mat_zoom, ZOOM, out=RESULTS / "zoom.png");

In [ ]:
hic_io.plot_pair(MCOOL, RES["zoom"], *PAIR, out=RESULTS / "pair.png");

## 4. Curva P(s)

Probabilità di contatto media in funzione della distanza, su tutto il cromosoma. `expected_cis` lavora a blocchi, quindi non carica la matrice intera.
La pendenza locale (d log P / d log s) intorno a −1 tra 100 kb e 1 Mb è tipica dell'interfase. Un picco/spalla della pendenza tra 100 e 300 kb indica i loop di cohesina.

In [ ]:
clr_ps = hic_io.open_cooler(MCOOL, RES["zoom"])
cvd = analysis.contact_probability(clr_ps, CHROM, nproc=NPROC)

fig, (ax_p, ax_s) = plt.subplots(2, 1, figsize=(5, 6), sharex=True, constrained_layout=True,
                                 gridspec_kw=dict(height_ratios=[2, 1]))
analysis.plot_contact_probability(cvd, ax=ax_p)
ax_p.set_xlabel("")
ax_s.semilogx(cvd["s_bp"], cvd["slope"])
ax_s.axhline(-1, color="grey", lw=0.8, ls="--")
ax_s.set(xlabel="distanza genomica s (bp)", ylabel="pendenza", ylim=(-2, 0.5))
cvd[["s_bp", "P", "slope"]].dropna().iloc[::10].head()

## 5. Compartimenti A/B

Primo autovettore (E1) della matrice O/E cis. Il segno è arbitrario, quindi lo orientiamo con il **contenuto GC** di hg38: E1 > 0 = A (ricco in GC, attivo).

In [ ]:
clr_c = hic_io.open_cooler(MCOOL, RES["compartments"])
if not DEMO and FASTA.exists():
    phasing = analysis.gc_phasing_track(clr_c, str(FASTA), CHROM)
elif DEMO:
    bins = clr_c.bins()[:][["chrom", "start", "end"]]
    phasing = bins.assign(value=true_compartment(bins["start"]))  # verità nota del sintetico
else:
    phasing = None  # senza genoma il segno di E1 è arbitrario

eigs = analysis.compartments(clr_c, phasing, CHROM)
eigs["compartment"].value_counts(dropna=False)

In [ ]:
mat_c = hic_io.fetch_region(MCOOL, RES["compartments"], chrom_region)
e1 = eigs[eigs["chrom"] == CHROM]
x = (e1["start"] + e1["end"]) / 2e6

fig, (ax_m, ax_e) = plt.subplots(2, 1, figsize=(6, 7.5), sharex=True, constrained_layout=True,
                                 gridspec_kw=dict(height_ratios=[4, 1]))
hic_io.plot_matrix(mat_c, chrom_region, ax=ax_m, colorbar=False, title=f"{CHROM}: compartimenti A/B")
ax_m.set_aspect("auto")
ax_m.set_xlabel("")
ax_e.fill_between(x, e1["E1"], where=e1["E1"] > 0, color="tab:red", step="mid", label="A")
ax_e.fill_between(x, e1["E1"], where=e1["E1"] < 0, color="tab:blue", step="mid", label="B")
ax_e.set(xlabel=f"{CHROM} (Mb)", ylabel="E1")
ax_e.legend(loc="upper right", fontsize="small")
fig.savefig(RESULTS / "compartments.png", dpi=150)

## 6. Insulation e confini di TAD

Insulation score a diamante su più finestre (5, 10 e 25 bin). I confini sono i minimi locali con forza sopra la soglia di Li, calcolati sulla finestra di 10 bin.

In [ ]:
clr_i = hic_io.open_cooler(MCOOL, RES["insulation"])
ins = analysis.insulation(clr_i, regions=INS_REGION, nproc=NPROC)
w = RES["insulation"] * 10
bounds = analysis.tad_boundaries(clr_i, window_bp=w, insulation_df=ins)
print(f"{len(bounds)} confini in {INS_REGION} (finestra {w // 1000} kb)")
bounds.head()

In [ ]:
ins_region = cooler.util.parse_region(INS_REGION, clr_i.chromsizes)
mat_i = hic_io.fetch_region(MCOOL, RES["insulation"], ins_region)

fig, (ax_m, ax_i) = plt.subplots(2, 1, figsize=(6, 7.5), sharex=True, constrained_layout=True,
                                 gridspec_kw=dict(height_ratios=[4, 1]))
hic_io.plot_matrix(mat_i, ins_region, ax=ax_m, colorbar=False, title=f"{INS_REGION}: TAD")
ax_m.set_aspect("auto")
ax_m.set_xlabel("")
mid = (ins["start"] + ins["end"]) / 2e6
for win in sorted({c.rsplit("_", 1)[1] for c in ins.columns if c.startswith("log2_insulation_score_")}, key=int):
    ax_i.plot(mid, ins[f"log2_insulation_score_{win}"], lw=1, label=f"{int(win) // 1000} kb")
for b in (bounds["start"] + bounds["end"]) / 2e6:
    ax_m.axvline(b, color="tab:blue", lw=0.6, alpha=0.6)
    ax_i.axvline(b, color="tab:blue", lw=0.6, alpha=0.6)
ax_i.set(xlabel=f"{ins_region[0]} (Mb)", ylabel="log2 insulation")
ax_i.legend(fontsize="small", ncol=3)
fig.savefig(RESULTS / "insulation.png", dpi=150)

## 7. Vista circolare (Circos)

Il genoma intero su un cerchio, con i cromosomi come segmenti bianchi e neri.
- **Corde magenta (trans):** coppie di bin di cromosomi diversi con contatti più alti della media di quella coppia di cromosomi.
- **Petali arancioni (cis a lungo raggio, ≥ 10 Mb):** coppie di bin dello stesso cromosoma più alte dell'atteso per la loro distanza, cioè la media della diagonale. Tipicamente sono blocchi A lontani che si cercano.
- **In verde** le corde che toccano il cromosoma in esame. L'anello interno mostra E1 genome-wide (rosso = A, blu = B).

Serve il genoma intero, quindi la risoluzione è bassa: 1 Mb, circa 3 100 bin e 76 MB.
Il dataset sintetico non ha contatti trans e i suoi cromosomi sono più corti di 10 Mb: nella demo le corde cis usano 400 kb come distanza minima.

In [ ]:
clr_g = hic_io.open_cooler(MCOOL, RES["circos"])
chroms = list(analysis.make_view(clr_g)["chrom"])
min_cis = 400_000 if DEMO else 10_000_000
links = pd.concat([trans_links(clr_g, chroms, top_n=1000),
                   cis_links(clr_g, chroms, min_distance=min_cis, top_n=300)], ignore_index=True)

if not DEMO and FASTA.exists():
    phasing_g = analysis.gc_phasing_track(clr_g, str(FASTA))
elif DEMO:
    bins_g = clr_g.bins()[:][["chrom", "start", "end"]]
    phasing_g = bins_g.assign(value=true_compartment(bins_g["start"]))
else:
    phasing_g = None
eigs_g = analysis.compartments(clr_g, phasing_g)

plot_circos(clr_g.chromsizes[chroms], links, track=eigs_g, highlight=CHROM,
            out=RESULTS / "circos.png", title=f"{MCOOL.stem}: contatti trans e cis a lungo raggio");
is_cis = links["chrom1"] == links["chrom2"]
print(f"{(~is_cis).sum()} corde trans, {is_cis.sum()} cis")

## 8. Salvataggio dei risultati

In [ ]:
tag = "synthetic" if DEMO else MCOOL.stem
outputs = {"ps": cvd, "compartments": eigs, "insulation": ins, "tad_boundaries": bounds,
           "circos_links": links}
for name, df in outputs.items():
    path = RESULTS / f"{tag}_{CHROM}_{name}.tsv"
    df.to_csv(path, sep="\t", index=False)
    print(f"{path.relative_to(ROOT)}: {len(df):,} righe")